# 04 — Model evaluation and annual scoring contract

This is a reusable, read-only model-review notebook. The retained hurdle is a continuous comparative research artifact. It loads saved artifacts and recorded final-test predictions; it does **not** tune, fit, score, or overwrite a model.

The output is an estimated next-year burned share, not a probability, safety score, or purchase recommendation.

## What this notebook is for

Use it after the documented project reproduction has completed to inspect the exact model contract, artifact metadata, final-temporal-test metrics, prediction diagnostics, and limitations. The figures are rendered in the notebook only; stable presentation figures are maintained separately by `src/final_visuals.py`.

In [ ]:
from pathlib import Path
import sys
import joblib
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_contract import TARGET_COLUMN
from src.feature_contract import PREDICTOR_COLUMNS
from src.notebook_reporting import (
    binned_observed_estimated_table,
    model_comparison_frame,
    model_component_frame,
    plot_binned_observed_estimated,
    plot_metric_comparison,
    plot_prediction_diagnostics,
)
from src.model_diagnostics import build_model_diagnostics, validate_model_diagnostics
from src.notebook_support import read_json_artifact, require_artifacts, resolve_project_root

PROJECT_ROOT = resolve_project_root(PROJECT_ROOT)
metrics = read_json_artifact(PROJECT_ROOT, 'data/processed/extended_model_selection_2010_2021/final_temporal_test_metrics.json')
selection_model_path, operational_model_path, predictions_path = require_artifacts(PROJECT_ROOT, [
    'data/processed/extended_model_selection_2010_2021/models/nine_feature_hurdle.joblib',
    'data/processed/final_model_2010_2024/nine_feature_hurdle.joblib',
    'data/processed/extended_model_selection_2010_2021/final_temporal_test_predictions.parquet',
])
selection_payload = joblib.load(selection_model_path)
operational_payload = joblib.load(operational_model_path)
predictions = pd.read_parquet(predictions_path)
print('Loaded recorded final-test evidence and saved model artifacts without fitting or scoring.')

## Optional controlled regeneration

Set the switch below to `True` only when you deliberately want to regenerate diagnostic figures/tables from the existing frozen final-test artifacts. It never fits a model. Full refitting and frozen final evaluation remain controlled by the one-command project runner because they are long-running stages with strict temporal safeguards.

In [ ]:
REBUILD_MODEL_DIAGNOSTICS = False  # Change deliberately; writes only reports/figures and reports/tables.
diagnostic_inventory = build_model_diagnostics() if REBUILD_MODEL_DIAGNOSTICS else validate_model_diagnostics()
display(pd.DataFrame([{'artifact type': key, 'path': path} for key, records in diagnostic_inventory.items() if key in {'figures', 'tables'} for path in records.values()]))
print('Diagnostic mode:', 'rebuilt' if REBUILD_MODEL_DIAGNOSTICS else 'verified existing')

## Split integrity and feature contract

Candidate selection was restricted to T=2010–2021. The frozen final temporal test is T=2022–2024; its outcome years are 2023–2025. The later operational artifact has the same selected specification but was refit after final evaluation through observed outcome 2025.

In [ ]:
assert metrics['design']['final_test_years'] == [2022, 2023, 2024]
assert metrics['design']['tuning_performed'] is False
assert selection_payload['feature_order'] == list(PREDICTOR_COLUMNS)
assert operational_payload['feature_order'] == list(PREDICTOR_COLUMNS)
assert set(predictions['observation_year']) == {2022, 2023, 2024}
assert (predictions['outcome_year'] == predictions['observation_year'] + 1).all()

contract = pd.DataFrame({'feature': PREDICTOR_COLUMNS, 'role': ['predictor'] * len(PREDICTOR_COLUMNS)})
display(contract)
display(pd.DataFrame([
    {'artifact': 'frozen final-test model', 'predictor_years': selection_payload['train_years'] + selection_payload['validation_years'], 'target': TARGET_COLUMN, 'random_seed': selection_payload['random_seed']},
    {'artifact': 'current operational refit', 'predictor_years': operational_payload['training_predictor_years'], 'target': operational_payload['target'], 'random_seed': operational_payload['random_seed']},
]))

## Model structure

The hurdle model multiplies two learned quantities: estimated occurrence of any burned share and estimated burned share conditional on fire. This is appropriate for a target with many exact zeros and continuous positive values. It is not a probability model offered to a buyer.

In [ ]:
component_summary = model_component_frame(selection_payload)
display(component_summary)
print('Final estimate = occurrence component × positive-share component.')
print('No feature-importance claim is shown: the saved histogram-gradient estimators have no native, directly comparable importance measure, and correlated spatial predictors require a separately designed interpretation analysis.')

## Final temporal test: metrics and comparison

MAE and RMSE are lower-is-better error measures. Positive-row measures focus on observed burned cells. Capture@20% is a technical ranking diagnostic: the share of positive-target cells found in the highest-ranked 20% of estimates. It is not a property-selection threshold.

In [ ]:
comparison = model_comparison_frame(metrics)
display(comparison.style.format({column: '{:.4f}' for column in comparison.columns if column != 'rows'}))
plot_metric_comparison(comparison)

In [ ]:
year_rows = []
for model_name, result in metrics['metrics'].items():
    for year, values in result['by_final_test_year'].items():
        year_rows.append({'model': model_name, 'predictor_year': int(year), 'MAE': values['mae_all'], 'RMSE': values['rmse_all'], 'positive_row_MAE': values['mae_positive'], 'capture_at_20_percent': values['capture_at_20_percent']})
by_year = pd.DataFrame(year_rows).sort_values(['predictor_year', 'model'])
display(by_year.style.format({'MAE': '{:.4f}', 'RMSE': '{:.4f}', 'positive_row_MAE': '{:.4f}', 'capture_at_20_percent': '{:.2%}'}))

## Prediction diagnostics

These visuals use the saved final-test prediction table. The first panel retains zero-target cells; the second isolates observed positive shares; the third shows estimation residuals by T. They are diagnostic evidence, not a forecast map.

In [ ]:
plot_prediction_diagnostics(predictions, model_column='nine_feature_hurdle')

In [ ]:
binned = binned_observed_estimated_table(predictions, model_column='nine_feature_hurdle')
display(binned.style.format({'mean_estimated_share': '{:.4f}', 'mean_observed_share': '{:.4f}', 'positive_target_share': '{:.2%}'}))
plot_binned_observed_estimated(binned)

## Durable diagnostic figures

These are stable report artifacts generated from the same frozen final-test predictions used above. They can be opened outside Jupyter and are recreated by `scripts/build_model_diagnostics.py`.

In [ ]:
from IPython.display import Image
for name, relative_path in diagnostic_inventory['figures'].items():
    path = PROJECT_ROOT / relative_path
    print(name, '—', relative_path)
    display(Image(filename=str(path), width=900))

## Optional model-stage orchestration

The notebook can call the same reusable refit and frozen-final-test functions used by the command-line runner. This is disabled by default because it overwrites derived model/evaluation artifacts and can take time. Use it only after a deliberate clean rebuild and never to experiment with final-test years.

In [ ]:
RUN_FROZEN_MODEL_STAGES = False  # Deliberate opt-in: refits T=2010–2021 and reruns the frozen T=2022–2024 evaluation.
if RUN_FROZEN_MODEL_STAGES:
    from src.extended_model_refit import run as refit_frozen_candidates
    from src.extended_final_test import run as run_frozen_final_test
    refit_result = refit_frozen_candidates()
    final_test_result = run_frozen_final_test()
    diagnostic_inventory = build_model_diagnostics()
    print('Rebuilt frozen model stages and durable diagnostics. Review the recorded reports before using results.')
else:
    print('Disabled by default. Preferred full automation: python scripts/run_project.py --mode reproduce --confirm-rebuild')

## Annual operational use and limits

For forecast year `Y`, build an unlabelled T=`Y−1` feature matrix and score it using a model refit only through labelled predictor year `Y−2`. When ICNF later supplies the outcome for `Y`, evaluate that score and refit the unchanged selected specification for `Y+1`.

Use `python scripts/prepare_operational_forecast.py` and `python scripts/score_operational_forecast.py` for the controlled annual cycle, or `python scripts/run_project.py --mode reproduce --confirm-rebuild` for full reproduction. Do not manually change the feature order, model parameters, or output interpretation in this notebook.